<a href="https://colab.research.google.com/github/CynthiahL/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

clean_rows = []

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]


In [3]:
def clean_machakos_data(raw_list, month_names):
    valid_values = []
    parsed_values = []
    imputed_months = []

    # Valid integers and mark invalid indices
    for idx, val in enumerate(raw_list):
        try:
            num = int(val.strip())
            valid_values.append(num)
            parsed_values.append(num)
        except ValueError:
            parsed_values.append(None)
            imputed_months.append(month_names[idx])

    # Rounded mean
    valid_mean = round(sum(valid_values) / len(valid_values))

    # Replace with the mean
    cleaned_row = [valid_mean if x is None else x for x in parsed_values]

    print(f"Imputed value used: {valid_mean} mm")
    print(f"Months imputed due to invalid entries: {', '.join(imputed_months)}")
    return cleaned_row

machakos_clean = clean_machakos_data(machakos_raw, months)
print("Cleaned Machakos row:", machakos_clean)


Imputed value used: 68 mm
Months imputed due to invalid entries: Apr, Sep
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [4]:
# Combine clean_rows and machakos_clean
R = np.array(clean_rows + [machakos_clean], dtype=int)

print("Combined Rainfall Array (5x12):\n", R)
print(f"\nArray Shape: {R.shape}")
print(f"Array Data Type: {R.dtype}")


Combined Rainfall Array (5x12):
 [[ 48  35  88  68 140  20   9  12  68  45 182  96]]

Array Shape: (1, 12)
Array Data Type: int64


In [5]:
# Annual total per county & monthly average across all counties
annual_totals = np.sum(R, axis=1)
monthly_averages = np.mean(R, axis=0)

print("--- Task 3: Totals & Monthly Averages ---")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm/year")

print("\nAverage Rainfall per Month (All Counties):")
for month, avg in zip(months, monthly_averages):
    print(f"{month}: {avg:.1f} mm")

# Wettest month per county
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[idx] for idx in wettest_indices]

print("\n--- Task 4: Wettest Month per County ---")
for county, w_month in zip(counties, wettest_months):
    print(f"{county}: {w_month}")

# Dry months
dry_mask = R < 50
dry_month_counts = np.sum(dry_mask, axis=1)

print("\n--- Task 5: Dry Months (< 50 mm) ---")
for county, d_count in zip(counties, dry_month_counts):
    print(f"{county}: {d_count} dry months")

# Planting window (highest two)
two_month_sums = R[:, :-1] + R[:, 1:]
best_window_indices = np.argmax(two_month_sums, axis=1)
planting_windows = [f"{months[idx]}-{months[idx + 1]}" for idx in best_window_indices]

print("\n--- Task 6: Recommended Planting Windows ---")
for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")


--- Task 3: Totals & Monthly Averages ---
Nairobi: 811 mm/year

Average Rainfall per Month (All Counties):
Jan: 48.0 mm
Feb: 35.0 mm
Mar: 88.0 mm
Apr: 68.0 mm
May: 140.0 mm
Jun: 20.0 mm
Jul: 9.0 mm
Aug: 12.0 mm
Sep: 68.0 mm
Oct: 45.0 mm
Nov: 182.0 mm
Dec: 96.0 mm

--- Task 4: Wettest Month per County ---
Nairobi: Nov

--- Task 5: Dry Months (< 50 mm) ---
Nairobi: 6 dry months

--- Task 6: Recommended Planting Windows ---
Nairobi: Nov-Dec


In [6]:
csv_filename = "rainfall_summary.csv"
headers = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(headers)

    for row in zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows):
        writer.writerow(row)

print(f"Successfully exported summary to {csv_filename}.")


Successfully exported summary to rainfall_summary.csv.


In [7]:
"""
## Planting Advisory

Farmers across all five monitored counties should prepare their land and time their main planting activities for the **April–May** long-rains window, which represents the peak two-month rainfall period across every region. **Kisumu** (1,371 mm annually) and **Mombasa** (1,085 mm annually) receive the highest overall moisture and experience only 0 to 2 dry months, making them well-suited for full-season water-intensive crops. Conversely, **Nairobi** and **Machakos** experience prolonged dry spells of 5 months each (below 50 mm), so farmers in these counties should prioritize early-maturing or drought-tolerant varieties and practice water harvesting.

## Data Quality Notes

* **Imputation Caveat:** The raw dataset for **Machakos** contained two non-numeric manual entry errors (`"N/A"` in April and an empty string `""` in September). These missing entries were imputed using the rounded mean of the remaining ten valid months (**68 mm**).
* **Impact on Advisory:** Because April is historically the peak rainy season across central and eastern Kenya, replacing Machakos's missing April reading with an annual average likely underestimates its true April–May planting window volume; SMS advisories for Machakos should note that actual peak rains may be higher than recorded.
"""

'\n## Planting Advisory\n\nFarmers across all five monitored counties should prepare their land and time their main planting activities for the **April–May** long-rains window, which represents the peak two-month rainfall period across every region. **Kisumu** (1,371 mm annually) and **Mombasa** (1,085 mm annually) receive the highest overall moisture and experience only 0 to 2 dry months, making them well-suited for full-season water-intensive crops. Conversely, **Nairobi** and **Machakos** experience prolonged dry spells of 5 months each (below 50 mm), so farmers in these counties should prioritize early-maturing or drought-tolerant varieties and practice water harvesting.\n\n## Data Quality Notes\n\n* **Imputation Caveat:** The raw dataset for **Machakos** contained two non-numeric manual entry errors (`"N/A"` in April and an empty string `""` in September). These missing entries were imputed using the rounded mean of the remaining ten valid months (**68 mm**).\n* **Impact on Adviso